# L0 · LLM 调用协议与会话记忆

**目标：** 从调用角度看清 Chat Completions 协议，并亲手验证「会话记忆 = 回灌 messages」。

| 练习 | 通关标准 |
|------|----------|
| 1 单轮调用 | 能打印 `role` / `content` / `finish_reason` |
| 2 system 角色 | 换 system 后回答风格明显变化 |
| 3 失忆对照 | A 无历史答非所问；B 有历史能续聊 |
| 4 最小会话环 | 自己维护 `messages` 列表完成两轮 |

本册自包含，不跳转到其他脚本。

## 0. 环境准备

使用通用 OpenAI-compatible 客户端；Base URL 与模型名全部从 `.env` 读取，不绑定供应商。

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

from dotenv import load_dotenv


def find_repo_root() -> Path:
    """向上找到含 pyproject.toml 的仓库根，避免 notebook 工作目录不在根上。"""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
print("cwd =", ROOT)
print(
    "model_configured =",
    bool(os.getenv("LLM_BASE_URL") and os.getenv("LLM_MODEL")),
)

from openai import OpenAI

base = (os.getenv('LLM_BASE_URL') or '').strip()
model = (os.getenv('LLM_MODEL') or '').strip()
assert base and model, '请先在 .env 填写 LLM_BASE_URL 和 LLM_MODEL'

client = OpenAI(api_key=os.getenv('LLM_API_KEY') or 'not-required', base_url=base)
MODEL = model
print('base_url =', base)
print('model    =', MODEL)

## 1. 单轮调用：看清请求与响应形状

协议核心：`POST {base_url}/chat/completions`，body 里带 `model` + `messages`。

**任务：** 发起一次调用，打印 assistant 的 `role`、`content`、以及 `finish_reason`。

In [ ]:
import json

messages = [
    {'role': 'system', 'content': '你是 OOCL 订舱助手。回答简洁，不超过两句话。'},
    {'role': 'user', 'content': '用一句话说明：什么是订舱预审？'},
]

print('=== 请求 messages ===')
print(json.dumps(messages, ensure_ascii=False, indent=2))

resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
choice = resp.choices[0]
msg = choice.message

print('\n=== 响应关键字段 ===')
print('role         :', msg.role)
print('finish_reason:', choice.finish_reason)
print('content      :', msg.content)
if resp.usage:
    print('usage        :', resp.usage.model_dump())

## 2. system 角色：宪法如何改变行为

同一 `user` 问题，换 `system`，观察输出差异。

**任务：** 对比「严格只答 Yes/No」与「订舱受理管理员：三条要点 + 例外提醒」两种 system。


In [ ]:
question = '锂电池订舱必须提供 MSDS 吗？'

def ask(system: str, user: str) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
    )
    return r.choices[0].message.content or ''

strict = ask('你只能回答 Yes 或 No，禁止解释。', question)
tutor = ask('你是订舱受理管理员，用三条要点解释，并提醒例外情况。', question)

print('[strict]\n', strict)
print('\n[tutor]\n', tutor)
print('\n观察：同样的 user，不同的 system → 协议层角色确实在起作用。')


## 3. 会话记忆对照：失忆 vs 续聊

LLM **无状态**。第二轮若只发新问题、不带历史，模型看不到「锂电池 / 上海港」语境。

**任务：** 跑 A（无历史）与 B（带历史），对比答案。

In [ ]:
system = {'role': 'system', 'content': '你是订舱规则助手。若信息不足，明确说缺少上文。'}

user1 = {'role': 'user', 'content': '上海港对锂电池订舱有什么文件要求？请列 2–3 条。'}
r1 = client.chat.completions.create(
    model=MODEL, temperature=0, messages=[system, user1]
)
assistant1 = {
    'role': 'assistant',
    'content': r1.choices[0].message.content or '',
}
print('【第1轮回答】\n', assistant1['content'][:500], '\n')

follow_up = {'role': 'user', 'content': '那洛杉矶港呢？'}

# A：失忆——只发追问
r_a = client.chat.completions.create(
    model=MODEL, temperature=0, messages=[system, follow_up]
)
print('【A · 无历史】\n', r_a.choices[0].message.content, '\n')

# B：续聊——回灌完整 messages
history = [system, user1, assistant1, follow_up]
r_b = client.chat.completions.create(model=MODEL, temperature=0, messages=history)
print('【B · 有历史】\n', r_b.choices[0].message.content)
print('\n通关检查：A 往往不知道「那」指锂电池规则；B 能接着上海港语境答洛杉矶。')

## 4*. 最小会话循环（你来维护 messages）

把「追加 user → 调用 → 追加 assistant」写成函数，完成两轮对话。

**通关：** `history` 长度应为 5（system + u1 + a1 + u2 + a2）。

In [ ]:
def chat_turn(history: list[dict], user_text: str) -> list[dict]:
    """在 history 末尾追加一轮 user/assistant，返回同一列表。"""
    history.append({'role': 'user', 'content': user_text})
    resp = client.chat.completions.create(
        model=MODEL, temperature=0, messages=history
    )
    reply = resp.choices[0].message
    history.append({'role': reply.role, 'content': reply.content or ''})
    return history


history = [
    {'role': 'system', 'content': '你是简洁的订舱助手。每轮回答不超过三句。'}
]
chat_turn(history, '危险品订舱一般要准备哪些单证？')
chat_turn(history, '如果缺 MSDS 会怎样？')

print('messages 条数 =', len(history))
assert len(history) == 5, '应为 system + 两轮 user/assistant'
for i, m in enumerate(history):
    preview = (m.get('content') or '')[:80].replace('\n', ' ')
    print(f"{i}. {m['role']:10s} | {preview}")

print('\n✅ L0 协议练习完成：你已经用调用视角实现了会话记忆。')

<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 完成一次 system + user 的单轮调用并查看响应结构。
- [ ] 对比两个不同 system 指令对同一问题的影响。
- [ ] 证明“不传历史会失忆、传入历史可续聊”。
- [ ] 完成最小多轮会话循环并检查 messages 顺序。
- [ ] 写下本页的通关口令。

**交付证据：**两组 system 对照、多轮 messages、tool call JSON 结构。